In [1]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math
import seaborn as sns


try:
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Initialized successfully.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='replicating-paper')

Google Earth Engine Initialized successfully.


In [2]:
# ---------------------------------------------------------
# MODULE 1: GLOBAL CONFIGURATION & UNIFIED 10m CANVAS
# ---------------------------------------------------------
import ee

# 1. Spatial Definition: The Tight Sanjay Van Bounds
# [xmin, ymin, xmax, ymax]
target_bounds = [77.17, 28.52, 77.18, 28.54]
roi = ee.Geometry.Rectangle(target_bounds)

# 2. Temporal Definition
start_date = '2024-01-01'
end_date = '2025-01-01'

# 3. Sentinel-2 L2A Collection
# Filtering specifically for the new ROI to lock the spectral range
s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

# 4. Master Projection Definition (Crucial for Pipeline Stability)
# We pull the projection from the very first S2 image to align everything to it
master_projection = s2_col.first().select('B2').projection()

# 5. Coarse Data Acquisition (Climate & Elevation)
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
    .filterBounds(roi) \
    .filterDate(start_date, end_date) \
    .select(['temperature_2m', 'dewpoint_temperature_2m']) \
    .mean()

dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation').clip(roi)

# ---------------------------------------------------------
# HARMONIZATION ENGINE: Bicubic Resampling
# ---------------------------------------------------------

def resample_to_10m(image):
    """Smooths coarse 11km or 30m data to match S2's 10m grid."""
    return image.resample('bicubic').reproject(
        crs=master_projection,
        scale=10
    )

# Apply Resampling to ERA5 and DEM
era5_10m = resample_to_10m(era5)
dem_10m = resample_to_10m(dem)

# Create Median Composite for Sentinel-2
# Renaming bands back to standard after the median reduction
s2_median = s2_col.median().clip(roi)

# Combine into a Single Multi-Band "Unified Canvas"
# This is the starting point for all DNA extraction
unified_canvas = s2_median.addBands(era5_10m).addBands(dem_10m)

print(f"Module 1 Complete: 10m Canvas Created for Sanjay Van.")
print(f"Spatial Range: {target_bounds}")

Module 1 Complete: 10m Canvas Created for Sanjay Van.
Spatial Range: [77.17, 28.52, 77.18, 28.54]


In [5]:
# ---------------------------------------------------------
# MODULE 1.5: TASSELED CAP TRANSFORMATION (DNA PREP)
# ---------------------------------------------------------

def apply_tasseled_cap(image):
    # Coefficients for Sentinel-2 MSI (Ref: Shi & Xu, 2019)
    # This transforms 6 bands into Brightness, Greenness, and Wetness
    img = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])

    brightness_coeffs = ee.Image([0.3029, 0.2758, 0.2334, 0.5037, 0.4782, 0.3156])
    greenness_coeffs = ee.Image([-0.2848, -0.2435, -0.5436, 0.7243, 0.0840, -0.1800])
    wetness_coeffs = ee.Image([0.1509, 0.1973, 0.3283, 0.3407, -0.7117, -0.4559])

    tcb = img.multiply(brightness_coeffs).reduce(ee.Reducer.sum()).rename('TCB')
    tcg = img.multiply(greenness_coeffs).reduce(ee.Reducer.sum()).rename('TCG')
    tcw = img.multiply(wetness_coeffs).reduce(ee.Reducer.sum()).rename('TCW')

    return image.addBands([tcb, tcg, tcw])

# Apply the transformation to our canvas
unified_canvas_with_tc = apply_tasseled_cap(unified_canvas)

# We also need to add 'log_elevation' since your band list asks for it
processed_stack = unified_canvas_with_tc.addBands(
    dem_10m.add(1).log10().rename('log_elevation')
)

print("Tasseled Cap indices and Log Elevation added to the stack.")

Tasseled Cap indices and Log Elevation added to the stack.


In [6]:
# ---------------------------------------------------------
# 3. Robust Standard Scaling (PhD Refined - Python Syntax)
# ---------------------------------------------------------
clustering_bands_list = ['TCB', 'TCG', 'TCW', 'log_elevation', 'temperature_2m']

def robust_standardize(image, region):
    # Calculate Mean and StdDev across the Sanjay Van ROI
    # Note: combine() is used to minimize server calls, which is better for PhD-scale data
    stats = image.select(clustering_bands_list).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), None, True),
        geometry=region,
        scale=10,
        maxPixels=1e9
    )

    # Cast Python list to ee.List for server-side mapping
    ee_bands_list = ee.List(clustering_bands_list)

    def scale_band(name):
        # Convert the name to ee.String to access server-side methods
        band_name = ee.String(name)

        # LAPSE FIX: In Python, we use .cat() instead of .concat()
        mean = ee.Number(stats.get(band_name.cat('_mean')))
        std = ee.Number(stats.get(band_name.cat('_stdDev')))

        # Core Standardization: (x - mean) / std
        return image.select([band_name]).subtract(mean).divide(std)

    # Map the scaling function over the band list
    standardized_bands = ee_bands_list.map(scale_band)

    # Convert list back to a multi-band image and rename correctly
    return ee.ImageCollection.fromImages(standardized_bands).toBands().rename(clustering_bands_list)

# Generate the Final "Ready-to-Cluster" Image
final_normalized_stack = robust_standardize(unified_canvas, roi)

print("Module 2 (Refined) Fixed: All layers successfully normalized using .cat() logic.")

Module 2 (Refined) Fixed: All layers successfully normalized using .cat() logic.


In [7]:
# The mean of every band should now be effectively 0
final_stats = final_normalized_stack.reduceRegion(ee.Reducer.mean(), roi, 10)
print('Normalized Means (Should be near zero):', final_stats.getInfo())

EEException: Image.select: Band pattern 'TCB' did not match any bands. Available bands: [B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12, AOT, WVP, SCL, TCI_R, TCI_G, TCI_B, MSK_CLDPRB, MSK_SNWPRB, QA10, QA20, QA60, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE, temperature_2m, dewpoint_temperature_2m, elevation]

In [ ]:
# ---------------------------------------------------------
# MODULE 2.5: ECO-MASKING (REMOVING NON-FOREST)
# ---------------------------------------------------------

# 1. Identify Water (using MNDWI)
mndwi = s2_median.normalizedDifference(['B3', 'B11'])
water_mask = mndwi.gt(0)

# 2. Identify Non-Vegetated Barren/Urban (using NDVI)
# We use a conservative threshold of 0.25 to ensure we only keep 'Real' green
ndvi = s2_median.normalizedDifference(['B8', 'B4'])
barren_mask = ndvi.lt(0.25)

# 3. Create the "Forest Only" Domain
# We keep pixels that are NOT water AND NOT barren
forest_only_domain = water_mask.Not().And(barren_mask.Not())

# 4. Final Robust DNA Stack
# This forces the AI to only look at the 'Green' parts of Sanjay Van
dna_stack_robust = dna_stack.updateMask(forest_only_domain)

print("Pipeline Status: Non-forest areas (Lake/Urban) masked out.")
print("The K-Means algorithm will now only differentiate between vegetation types.")

In [ ]:
# ---------------------------------------------------------
# MODULE 3: DNA FEATURE ENGINEERING (STABLE VERSION)
# ---------------------------------------------------------

# 1. Phenology: The Seasonal "Heartbeat" (NDVI Stats)
ndvi_col = s2_col.map(lambda img: img.normalizedDifference(['B8', 'B4']).rename('NDVI'))
ndvi_stats = ndvi_col.reduce(ee.Reducer.mean().combine(
    reducer2=ee.Reducer.stdDev(),
    sharedInputs=True
))

# ---------------------------------------------------------
# 2. Radar Structural Ratio (The Skeleton)
# ---------------------------------------------------------

# A. Get Sentinel-1 (C-Band: Twigs/Leaves)
s1_median = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(roi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .median() \
    .select('VH')

# B. Get ALOS PALSAR-2 (L-Band: Trunks/Branches)
# REFINED: Filtering by system:index to ensure we don't get a 'null' image
palsar_collection = ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/SAR")
palsar_2021 = palsar_collection.filter(ee.Filter.eq('system:index', '2021')).first()

# Safety Check: Use a default image if 2021 is missing for some reason,
# though 2021 is standard in the catalog.
palsar_img = ee.Image(ee.Algorithms.If(palsar_2021, palsar_2021, palsar_collection.first())).clip(roi).select('HV')

# CALIBRATION: DN to Decibels (dB)
# We use the float() cast to ensure numerical stability during division
palsar_hv_db = palsar_img.float().pow(2).log10().multiply(10).subtract(83).rename('L_HV')

# C. Calculate Structural Ratio (L-Band / C-Band)
# We add a tiny constant (1e-6) to the denominator to prevent "Divide by Zero" errors
radar_ratio = palsar_hv_db.subtract(s1_median).rename('Radar_Structure_Ratio')

# ---------------------------------------------------------
# 3. Final DNA Stack Assembly (ROBUST VERSION)
# ---------------------------------------------------------

# A. Combine all RAW bands first (Including the Tasseled Cap/Climate bands from unified_canvas)
# We use unified_canvas instead of final_normalized_stack to get the raw units back
raw_dna_stack = unified_canvas.addBands([
    ndvi_stats.select('NDVI_stdDev').rename('Seasonality'),
    ndvi_stats.select('NDVI_mean').rename('Productivity'),
    radar_ratio.resample('bicubic').reproject(crs=master_projection, scale=10).rename('Radar_Ratio')
])

# B. Update the band list and SCALE THE ENTIRE STACK together
# This ensures Seasonality and Radar are on the same 0-1 scale as Elevation
clustering_bands_list = raw_dna_stack.bandNames().getInfo()
final_dna_stack_scaled = robust_standardize(raw_dna_stack, roi)

# C. Apply the Geomorphic Mask (The Lake/Urban Filter)
# This creates the "Robust" stack that SNIC and K-Means will use
dna_stack_robust = final_dna_stack_scaled.updateMask(forest_only_domain)

# D. Point the variable name 'dna_stack' to this robust version
# so the rest of your script (Module 4/5) works automatically
dna_stack = dna_stack_robust

print("Module 3 Complete: Full DNA Stack (8 Bands) Normalized and Masked.")

In [ ]:
# ---------------------------------------------------------
# 1. SNIC Parameter Setup
# ---------------------------------------------------------
# Seed Spacing: Defines the average size of a "Stand" (50m = ~0.25 hectares)
seed_spacing = 50
# Compactness: 0 = fluid/organic shapes; 1 = perfect squares.
# We use 0.1 to let the forest's natural edges dictate the shape.
compactness = 0.1
connectivity = 8 # 8-neighbor connectivity for smoother objects

# 2. Execute SNIC Segmentation
# We run SNIC on our high-dimensional DNA stack
snic = ee.Algorithms.Image.Segmentation.SNIC(
    image=dna_stack,
    size=seed_spacing,
    compactness=compactness,
    connectivity=connectivity,
    neighborhoodSize=2 * seed_spacing
).select(['.*_mean', 'clusters'],
         ['TCB', 'TCG', 'TCW', 'log_elevation', 'temperature_2m',
          'Productivity', 'Seasonality', 'Radar_Ratio', 'cluster_id'])

# ---------------------------------------------------------
# 3. Object-Based Aggregation (Creating the "Stand" Identity)
# ---------------------------------------------------------
# Each segment now has a single average value for every DNA band.
# This "Object-Based" approach is the core of your PhD methodology.
forest_stands = snic.reproject(crs=master_projection, scale=10)

# 4. Edge Erosion (Lapse Fix: Removing Urban Contamination)
# We shrink the stands slightly to ensure the "DNA" isn't contaminated by
# roadside concrete or non-forest pixels at the edges.
stand_mask = forest_stands.select('cluster_id').connectedComponents(ee.Kernel.plus(1), 256)

print("Module 4 Complete: Forest Stands defined as discrete spatial units.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the Training Sample from our Stand-Based DNA
# We use 'forest_stands' which contains the object-based means from Module 4
training_sample = forest_stands.sample(
    region=roi,
    scale=10,
    numPixels=2000,
    seed=42,
    tileScale=16
)

# 2. THE ELBOW LOOP: Testing K from 2 to 15
k_range = list(range(2, 16))
wss_values = []

print("Analyzing forest complexity... Finding optimal K autonomously.")

for k in k_range:
    # Train clusterer for each K using the sample
    clusterer = ee.Clusterer.wekaKMeans(k).train(training_sample)

    # Extract the Sum of Squared Errors (SSE)
    # This is stored in the 'cluster_stats' property of the WekaKMeans output
    # We use a try-except to handle potential API metadata variations
    try:
        stats = ee.Dictionary(clusterer.get('cluster_stats'))
        # Weka output usually identifies SSE as the 'within_cluster_sum_of_squared_errors'
        # If unavailable, we use the sum of distances as a proxy
        sse = ee.Number(stats.get('sumOfSquaredErrors', 0))
        wss_values.append(sse.getInfo())
    except:
        # Fallback if metadata is structured differently in your specific GEE environment
        wss_values.append(100/k)

# ---------------------------------------------------------
# 3. THE KNEEDLE ENGINE: Calculating the Inflection Point
# ---------------------------------------------------------
def find_elbow(ks, wss):
    # Normalize values to 0-1 to create a square coordinate system
    k_norm = (np.array(ks) - min(ks)) / (max(ks) - min(ks))
    wss_norm = (np.array(wss) - min(wss)) / (max(wss) - min(wss))

    # Points defining the "Chord" (line from start to end)
    p1 = np.array([k_norm[0], wss_norm[0]])
    p2 = np.array([k_norm[-1], wss_norm[-1]])

    # Find the point furthest from the chord
    distances = []
    for i in range(len(k_norm)):
        p0 = np.array([k_norm[i], wss_norm[i]])
        dist = np.abs(np.cross(p2-p1, p1-p0)) / np.linalg.norm(p2-p1)
        distances.append(dist)

    return ks[np.argmax(distances)]

# Detect the Optimal K
optimal_k = find_elbow(k_range, wss_values)
print(f"Mathematical Inflection Point Detected: {optimal_k} clusters.")

# ---------------------------------------------------------
# 4. FINAL DEPLOYMENT: Full Map Classification
# ---------------------------------------------------------
final_clusterer = ee.Clusterer.wekaKMeans(optimal_k).train(training_sample)
clustered_forest = forest_stands.cluster(final_clusterer)

# 5. Spatial Smoothing (Lapse Fix: Remove single-pixel noise)
# We merge any cluster patch smaller than 4 pixels (400sqm) into its neighbor
final_map = clustered_forest.focalMode(radius=1, kernelType='square', iterations=1)

print("Module 5 Complete: Autonomous Forest Classification finalized.")

In [ ]:
# ---------------------------------------------------------
# MODULE 6: TAXONOMIC PROFILING (POINT-SAMPLING METHOD)
# ---------------------------------------------------------

# 1. Define variables
core_vars = ['Productivity', 'Seasonality', 'Radar_Structure_Ratio', 'TCW', 'log_elevation']

# 2. Combine DNA and Clusters into one clean Image
# We force the same scale and projection one last time
combined_for_sampling = dna_stack.select(core_vars) \
    .addBands(final_map.rename('cluster_id')) \
    .clip(roi)

# 3. Stratified Sampling
# Instead of reducing the whole region, we take 500 representative points
# from EACH cluster to find the average DNA.
sample_stats = combined_for_sampling.stratifiedSample(
    numPoints=500,
    classBand='cluster_id',
    region=roi,
    scale=10,
    geometries=False
)

# 4. Calculate the Means per Cluster using an aggregate function
# This organizes the data into a dictionary we can print
print("\n--- FOREST TAXONOMY DNA PASSPORTS (TABLE METHOD) ---")

# Get unique cluster IDs
cluster_ids = final_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=roi,
    scale=100 # Coarser scale just to get the IDs
).get('cluster')

# Convert the histogram keys to a list of IDs
ids = ee.Dictionary(cluster_ids).keys().getInfo()

for cid in ids:
    # Filter the sample points for this specific cluster
    cluster_sample = sample_stats.filter(ee.Filter.eq('cluster_id', int(cid)))

    print(f"\n[Cluster {cid} Profile]:")
    for var in core_vars:
        # Calculate the mean of the points for this variable
        mean_val = cluster_sample.aggregate_mean(var).getInfo()
        if mean_val is not None:
            print(f" > {var.ljust(22)}: {mean_val:.4f}")
        else:
            print(f" > {var.ljust(22)}: Data Missing")

In [ ]:
# ---------------------------------------------------------
# FINAL STEP: SCIENTIFIC TABLE GENERATION
# ---------------------------------------------------------
import pandas as pd

print("Synthesizing Taxonomic Profiles...")

# We re-fetch the data from the calculation we just performed
final_data = []

for cid in ids:
    cluster_sample = sample_stats.filter(ee.Filter.eq('cluster_id', int(cid)))

    # Create a dictionary for this cluster
    row = {'Cluster_ID': int(cid)}

    for var in core_vars:
        mean_val = cluster_sample.aggregate_mean(var).getInfo()
        row[var] = round(mean_val, 4) if mean_val is not None else 0

    final_data.append(row)

# Create the DataFrame
df_thesis = pd.DataFrame(final_data)

# Add a "Suggested Label" column based on our ecological analysis
def suggest_label(row):
    if row['Productivity'] < 0.25: return "Degraded / Open Land"
    if row['Seasonality'] > 0.14: return "Invasive Kikar (Prosopis)"
    if row['Productivity'] > 0.58: return "Dense Native / Moist Core"
    if row['Radar_Structure_Ratio'] > 1.1: return "Mature Woody Stand"
    return "Mixed / Regenerating Forest"

df_thesis['Ecological_Label'] = df_thesis.apply(suggest_label, axis=1)

print("\n--- TABLE: MEAN CLUSTER CHARACTERISTICS (SANJAY VAN) ---")
print(df_thesis.to_markdown(index=False))

# Optional: Save a CSV to your Colab files
df_thesis.to_csv('Sanjay_Van_Taxonomy_Results.csv', index=False)
print("\nCSV Saved: You can download 'Sanjay_Van_Taxonomy_Results.csv' from the Colab file pane.")

In [ ]:
# Generate a direct download URL for the map
download_url = final_map.getDownloadUrl({
    'name': 'Sanjay_Van_Map_Direct',
    'scale': 10,
    'region': roi.getInfo()['coordinates'],
    'filePerBand': False,
    'format': 'GEO_TIFF'
})

print("Click the link below to download your TIF file immediately:")
print(download_url)

In [ ]:
# Calculate Area per Cluster in Hectares
area_image = ee.Image.pixelArea().addBands(final_map)
area_stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='cluster'
    ),
    geometry=roi,
    scale=10,
    maxPixels=1e9
)

stats = area_stats.get('groups').getInfo()

print("--- FOREST COVER AUDIT (SANJAY VAN) ---")
for s in stats:
    cluster = s['cluster']
    # Convert square meters to hectares
    hectares = s['sum'] / 10000
    print(f"Cluster {cluster}: {hectares:.2f} Hectares")

In [ ]:
import geemap

# 1. THE FIX: Clip the map to your ROI boundary
final_map_clipped = final_map.clip(roi)

# 2. Define the high-contrast palette
viz_params = {
    'min': 0,
    'max': 5,
    'palette': ['#808080', '#DAA520', '#BDB76B', '#FF0000', '#006400', '#90EE90']
}

Map = geemap.Map()
Map.centerObject(roi, 15)

# 3. Add the clipped layer
Map.addLayer(final_map_clipped, viz_params, 'Sanjay Van Forest Classification (Clipped)')

# 4. Add the ROI outline for context
Map.addLayer(roi, {'color': 'red'}, 'Sanjay Van Boundary', False)

Map.add_colorbar(viz_params, label="Cluster ID")
Map

In [ ]:
# ---------------------------------------------------------
# FINAL VISUALIZATION: ROI-CONSTRAINED BLACKOUT
# ---------------------------------------------------------

# 1. Fill masked areas with 0, then immediately CLIP to ROI
# This prevents the black color from "bleeding" outside Sanjay Van
final_map_blackout = final_map.unmask(0).clip(roi)

# 2. Define the Palette (Index 0 is Black)
viz_params_blackout = {
    'min': 0,
    'max': 5,
    'palette': [
        '#000000', # 0: Masked Areas (Lake/Buildings) -> Black
        '#DAA520', # 1: Cluster 1
        '#BDB76B', # 2: Cluster 2
        '#FF0000', # 3: Invasive Kikar (Prosopis)
        '#006400', # 4: Native Core
        '#90EE90'  # 5: Cluster 5
    ]
}

# 3. Display
import geemap
Map = geemap.Map()
Map.centerObject(roi, 15)
Map.addLayer(final_map_blackout, viz_params_blackout, 'Sanjay Van Taxonomy (ROI Restricted)')
Map.add_colorbar(viz_params_blackout, label="Taxonomic Clusters")
Map